In [1]:
!nvidia-smi

Thu Oct  8 07:04:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [4]:
import numpy as np
import time
from numba import cuda
# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out
# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [5]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Extrapolate or skip

    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [7]:
print(benchmark_run(100))

(0.06665800003702316, 4.0183419999948455, 0.15492599993649492)


In [9]:
print(benchmark_run(1000))

(0.49612099996920733, 0.7754539999496046, 0.21115100003044063)


In [10]:
print(benchmark_run(10000))

(2.676076999932775, 1.7538600000079896, 0.11437500006650225)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:
print(benchmark_run(100000))

(27.29100099986681, 0.7617460000801657, 0.07287700009328546)


In [12]:
print(benchmark_run(1000000))

(276.8324710000343, 6.101070999875446, 0.14879100012876734)


In [13]:
print(benchmark_run(5000000))

(nan, 13.771716999826822, 0.2744400001120084)


In [15]:
print(benchmark_run(10000000))

(nan, 27.75710099990647, 0.47160199983409257)


In [16]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0
# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [18]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0
# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # Exactly 5 blocks = 40 threads
log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [20]:
# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same clock cycle!
        d_total[0] += d_arr[idx]
# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        # HARDWARE ATOMIC LOCK: Hardware serializes updates to prevent collisions
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [24]:
N = 50000
EXPECTED_SUM = 50000.0
THREADS_PER_BLOCK = 256
BLOCKS = (N + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK

# Array containing 50,000 ones
arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(arr)
d_total = cuda.device_array(1, dtype=np.float32)

print("Expected sum:", EXPECTED_SUM)
print("\nNAIVE KERNEL: 5 consecutive trials")
print("-" * 55)
print(f"{'Trial':<10}{'Naive Output':<20}{'Deviation':<15}")

naive_results = []

for trial in range(1, 6):
    # Reset total before each trial
    d_total.copy_to_device(np.array([0.0], dtype=np.float32))

    naive_sum_kernel[BLOCKS, THREADS_PER_BLOCK](d_arr, d_total)
    cuda.synchronize()

    result = float(d_total.copy_to_host()[0])
    deviation = EXPECTED_SUM - result
    naive_results.append(result)

    print(f"{trial:<10}{result:<20.1f}{deviation:<15.1f}")

print("\nATOMIC KERNEL: comparison")
print("-" * 55)

d_total.copy_to_device(np.array([0.0], dtype=np.float32))
atomic_sum_kernel[BLOCKS, THREADS_PER_BLOCK](d_arr, d_total)
cuda.synchronize()

atomic_result = float(d_total.copy_to_host()[0])
print(f"Expected sum: {EXPECTED_SUM:.1f}")
print(f"Atomic output: {atomic_result:.1f}")
print(f"Atomic deviation: {EXPECTED_SUM - atomic_result:.1f}")

Expected sum: 50000.0

NAIVE KERNEL: 5 consecutive trials
-------------------------------------------------------
Trial     Naive Output        Deviation      
1         2.0                 49998.0        
2         2.0                 49998.0        
3         2.0                 49998.0        
4         2.0                 49998.0        
5         2.0                 49998.0        

ATOMIC KERNEL: comparison
-------------------------------------------------------
Expected sum: 50000.0
Atomic output: 50000.0
Atomic deviation: 0.0


In [25]:
import math
@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)

    if x < width and y < height:

      dx = float(x - center_x)
      dy = float(y - center_y)
      dist = math.sqrt(dx * dx + dy * dy)
      max_dist = math.sqrt(float(center_x**2 + center_y**2))
      intensity = 1.0 - (dist / max_dist)
      if intensity < 0.0:
        intensity = 0.0
      d_canvas[y, x] = intensity
# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")

Generated 2D matrix shape: (1024, 1024). Center value: 1.000


In [26]:

import numpy as np
import math
from numba import cuda

# 2D CUDA kernel
@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    # Get 2D coordinates
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))

        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity


# Image dimensions
WIDTH, HEIGHT = 1024, 1024

# Create a matrix and transfer it to GPU
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

# Configure 2D threads and blocks
threads_2d = (16, 16)

blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

# Launch kernel
radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas,
    WIDTH,
    HEIGHT,
    WIDTH // 2,
    HEIGHT // 2
)

cuda.synchronize()

# Copy results back to CPU
result_canvas = d_canvas.copy_to_host()

# Display verification results
print("Grid Dimensions:")
print("Blocks in X:", blocks_2d[0])
print("Blocks in Y:", blocks_2d[1])
print("Total Blocks:", blocks_2d[0] * blocks_2d[1])

print("\nGenerated 2D matrix shape:", result_canvas.shape)
print(f"Center pixel (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Outer corner pixel (0, 0): {result_canvas[0, 0]:.3f}")


Grid Dimensions:
Blocks in X: 64
Blocks in Y: 64
Total Blocks: 4096

Generated 2D matrix shape: (1024, 1024)
Center pixel (512, 512): 1.000
Outer corner pixel (0, 0): 0.000


In [29]:
import hashlib
def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    # Include GPU name
    h.update(cuda.get_current_device().name.encode('utf-8'))
    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest
# REPLACE WITH YOUR ACTUAL STUDENT ID AND EMPIRICAL CROSSOVER N* VALUE
STUDENT_ID = "230103231"
CROSSOVER_N = 10000 # Example: your value from Section 2
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 60FD74C1C16F1E3E


'60FD74C1C16F1E3E'